# LlamaIndex Tutorial 04: Hybrid Search (Vector + Keyword Retrieval)

This notebook demonstrates hybrid search combining semantic and keyword retrieval.

## What you'll learn:
- Combining vector and keyword search
- BM25 retrieval
- Fusion methods


In [ ]:
 !pip install llama-index llama-index-llms-groq llama-index-embeddings-huggingface  llama-index-retrievers-bm25 python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 14.1 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: llama-index-workflows
    Found existing installation: llama-index-workflows 2.14.1
    Uninstalling llama-index-workflows-2.14.1:
      Successfully uninstalled llama-index-workflows-2.14.1
  Attempting uninstall: llama-index-core
    Found existing installation: llama-index-core 0.14.14
    Uninstalling llama-index-core-0.14.14:━━━━━━━━━━━━━━━━━━━ 1/2 [llama-index-core]
      Successfully uninstalled llama-index-core-0.14.14━━━━━━━ 1/2 [llama-index-core]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [llama-index-core][llama-index-core]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index-vector-stores-qdrant 0.9.1 requires llama-index-core<0.15,>=0.13.0, but you have llama-index-core 0.12.52.post1 which is incompatible.


In [1]:
import os
from dotenv import load_dotenv
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("Please set GROQ_API_KEY in your .env file")

# Initialize Groq LLM
llm = Groq(model="llama-3.3-70b-versatile", temperature=0.7, api_key=groq_api_key)

# Initialize HuggingFace Embedding
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5", device="cpu")

Settings.llm = llm
Settings.embed_model = embed_model

print("✅ LLM and Embedding model initialized!")

/Users/rushikeshvinodkharche/opt/anaconda3/envs/meet/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ LLM and Embedding model initialized!


## Step 1: Setup Hybrid Retriever

In [3]:
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.node_parser import SimpleNodeParser

# Create documents
documents = [
    Document(text="Hybrid search combines vector and keyword search."),
    Document(text="BM25 is a ranking function for keyword search."),
    Document(text="Vector search uses semantic similarity."),
]

# Create index
index = VectorStoreIndex.from_documents(documents)

# Parse documents into nodes (required for BM25Retriever)
parser = SimpleNodeParser.from_defaults()
nodes = parser.get_nodes_from_documents(documents)

# Create vector retriever
vector_retriever = VectorIndexRetriever(index=index, similarity_top_k=2)

# Create BM25 retriever (now uses nodes instead of documents)
bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=2)

print("✅ Hybrid retrievers created!")

✅ Hybrid retrievers created!


## Step 2: Combine Retrievers

In [4]:
from llama_index.core.retrievers import QueryFusionRetriever

# Combine retrievers
retriever = QueryFusionRetriever(
    [vector_retriever, bm25_retriever],
    similarity_top_k=2,
    num_queries=1,
    mode="reciprocal_rerank"
)

# Create query engine
query_engine = RetrieverQueryEngine.from_args(retriever)

response = query_engine.query("What is hybrid search?")
print(response)

Hybrid search combines vector and keyword search.


## Summary

In this tutorial, we learned about hybrid search combining semantic and keyword retrieval.

### Key Takeaways:
- **Hybrid Search (Vector + Keyword Retrieval)** provides advanced RAG capabilities
- All examples use Groq LLM and HuggingFace embeddings
- Production-ready patterns and best practices

### Next Steps:
- Experiment with different configurations
- Try combining with other techniques
- Move to the next tutorial for more advanced concepts